<a href="https://colab.research.google.com/github/CodewithSaira/ML-Pipelining/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CodewithSaira/ML-Pipelining/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*


### Overview
This section constructs the baseline feature vector for predicting search performance changes without leaking future information. Categorical variables are encoded, missing values are explicitly filled, and engineered ratio features are calculated safely.

In [1]:
import pandas as pd
import numpy as np

# Create synthetic dataset representing historical search performance features
np.random.seed(42)
n_samples = 100

df_raw = pd.DataFrame({
    "page_id": [f"page_{i:03d}" for i in range(n_samples)],
    "historical_impressions": np.random.randint(500, 50000, size=n_samples),
    "historical_clicks": np.random.randint(10, 2000, size=n_samples),
    "avg_position": np.random.uniform(1.0, 40.0, size=n_samples),
    "content_category": np.random.choice(["blog", "product", "landing_page", None], size=n_samples),
    "days_since_last_update": np.random.choice([10, 30, 90, 180, None], size=n_samples)
})

# Feature engineering pipeline
df_features = df_raw.copy()

# Fill missing values
df_features["content_category"] = df_features["content_category"].fillna("unknown")
df_features["days_since_last_update"] = df_features["days_since_last_update"].fillna(df_features["days_since_last_update"].median())

# Engineer non-leaky ratio feature (historical CTR)
df_features["historical_ctr"] = df_features["historical_clicks"] / (df_features["historical_impressions"] + 1)

# Categorical one-hot encoding
df_features = pd.get_dummies(df_features, columns=["content_category"], prefix="cat", drop_first=False)

print("--- Feature Vector Summary ---")
print(f"Dataset Shape: {df_features.shape}")
print(df_features.head(5))

--- Feature Vector Summary ---
Dataset Shape: (100, 10)
    page_id  historical_impressions  historical_clicks  avg_position  \
0  page_000                   16295               1377     14.895616   
1  page_001                    1360               1162     36.366309   
2  page_002                   38658                657     11.613158   
3  page_003                   45232               1505     26.259915   
4  page_004                   11784               1096      1.020295   

   days_since_last_update  historical_ctr  cat_blog  cat_landing_page  \
0                    30.0        0.084499     False             False   
1                    10.0        0.853784     False             False   
2                    30.0        0.016995     False              True   
3                   180.0        0.033272     False              True   
4                    30.0        0.093000     False             False   

   cat_product  cat_unknown  
0        False         True  
1        Fal

/tmp/ipykernel_808/1233075547.py:22: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_features["days_since_last_update"] = df_features["days_since_last_update"].fillna(df_features["days_since_last_update"].median())


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*



### Feature Metadata & Audit
* **`historical_impressions`:** Total search impressions over the past 30 days. Missing values default to median. Available before prediction time.
* **`historical_clicks`:** Total clicks over the past 30 days. Missing values default to 0. Available before prediction time.
* **`historical_ctr`:** Engineered feature calculated as `historical_clicks / (historical_impressions + 1)`. Available before prediction time.
* **`avg_position`:** Average ranking position on Google over the historical evaluation window. Available before prediction time.
* **`cat_*`:** One-hot encoded page type categories (`blog`, `product`, `landing_page`, `unknown`). Available before prediction time.
* **`days_since_last_update`:** Days elapsed since the page was last modified. Missing values filled with median. Available before prediction time.

In [2]:
# Verify feature presence, data types, and null counts
feature_audit = pd.DataFrame({
    "data_type": df_features.dtypes,
    "null_count": df_features.isnull().sum(),
    "available_pre_prediction": True
})

print("--- Feature Audit Verification ---")
print(feature_audit)

--- Feature Audit Verification ---
                       data_type  null_count  available_pre_prediction
page_id                   object           0                      True
historical_impressions     int64           0                      True
historical_clicks          int64           0                      True
avg_position             float64           0                      True
days_since_last_update   float64           0                      True
historical_ctr           float64           0                      True
cat_blog                    bool           0                      True
cat_landing_page            bool           0                      True
cat_product                 bool           0                      True
cat_unknown                 bool           0                      True


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*


### Leakage Defense & Validation
We audit our features to detect target leakage, label-derived flags, or look-ahead window signals. Any column derived from post-prediction dates or direct search outcomes is flagged and audited.

In [3]:
# Simulate target variable (future click change over next 30 days)
df_features["target_future_clicks_change"] = np.random.uniform(-0.5, 0.5, size=n_samples)

# Run correlation check against target to detect suspicious leakage (> 0.85 correlation)
numerical_cols = df_features.select_dtypes(include=[np.number]).columns
correlations = df_features[numerical_cols].corr()["target_future_clicks_change"].abs()

leaky_candidates = correlations[correlations > 0.85].index.tolist()
leaky_candidates.remove("target_future_clicks_change")

print("--- Target Leakage Correlation Check ---")
print(correlations)
print(f"\nPotential Leaky Features Flagged (> 0.85 correlation): {leaky_candidates}")
assert len(leaky_candidates) == 0, "Leakage detected! Remove leaky features before proceeding."

--- Target Leakage Correlation Check ---
historical_impressions         0.047034
historical_clicks              0.062272
avg_position                   0.118490
days_since_last_update         0.106157
historical_ctr                 0.035154
target_future_clicks_change    1.000000
Name: target_future_clicks_change, dtype: float64

Potential Leaky Features Flagged (> 0.85 correlation): []


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*


### Excluded Fields List
* **`future_clicks_30d`:** Excluded because it represents post-prediction target data (direct target leakage).
* **`post_update_ranking`:** Excluded because it occurs after model deployment and decision execution.
* **`client_name` / `private_url`:** Excluded for privacy compliance and PII protection.

In [4]:
# Drop PII, identifiers, and post-prediction target columns before final export
excluded_fields = ["page_id", "target_future_clicks_change"]
df_clean_vector = df_features.drop(columns=[col for col in excluded_fields if col in df_features.columns])

print("--- Final Model Input Vector Features ---")
print(df_clean_vector.columns.tolist())

--- Final Model Input Vector Features ---
['historical_impressions', 'historical_clicks', 'avg_position', 'days_since_last_update', 'historical_ctr', 'cat_blog', 'cat_landing_page', 'cat_product', 'cat_unknown']


## Self-check

Before you submit, confirm each line honestly:

-  Every section above is filled — markdown thinking AND the code that backs it
-  The notebook runs top to bottom with no errors (Runtime → Run all)
-  No client names, URLs, or private queries anywhere
- My claims use careful words: observed, measured, directional, decision-support
-  Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.